# 08 Learning Curves

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb008_s5_learning_curve_trace.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_trace.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s5_presentation_curves.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_presentation_curves.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s5_train_val_curve.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_curve.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s5_learning_curve_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s5_train_val_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_trace.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'ccafb97d395e824b8635ffe4dc946da24a316612'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_presentation_curves.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'd752b0ea3a69cbbec2555d759935a5279881f8eb'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_curve.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'eb8995ed6be49aedfcd84cee241543df630e6b40'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'b72e2c1ce63372fc5b21fd79b9ffe18deb279e4c'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '491294a250899a59a162f09883ea32d81faf89cf'}}

## exp_cmb008_s5_learning_curve_trace

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_trace.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_trace.py)  
Git blob: `ccafb97d395e824b8635ffe4dc946da24a316612`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 / 12: faithful per-epoch S5 trace and continuation.

Variants per seed and held-out writer:
  es10       exact S5 ES=True settings; records stop and best checkpoint.
  internal120 ES=True, same train/validation split/optimizer, patience 200,
              continue to 120 epochs (NO Writer/OfficialVal early stopping).
  full120    ES=False, 100% training fold, patience 200, continue to 120 epochs.
Neither held-out Writer nor OfficialVal is used to stop or choose a model.
The 120-epoch

CLI 인자 (정적 추출): `--seed`, `--heldout`, `--variant`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_trace.py
#!/usr/bin/env python3
"""EXP-CMB-008 / 12: faithful per-epoch S5 trace and continuation.

Variants per seed and held-out writer:
  es10       exact S5 ES=True settings; records stop and best checkpoint.
  internal120 ES=True, same train/validation split/optimizer, patience 200,
              continue to 120 epochs (NO Writer/OfficialVal early stopping).
  full120    ES=False, 100% training fold, patience 200, continue to 120 epochs.
Neither held-out Writer nor OfficialVal is used to stop or choose a model.
The 120-epoch limit is for diagnosis, not a newly selected training HP.
"""
from __future__ import annotations
import argparse,csv,json,time,warnings
from pathlib import Path
import numpy as np
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

from exp_cmb008_s5_early_stopping import (
    ROOT,WRITERS,LABELS,V3_SHA,HANDMADE_SHA,load_official,load_writers,
    PixelScaler255,prepareraw,process
)
PAIRS=((13,0,"D_to_0"),(11,8,"B_to_8"),(8,11,"8_to_B"),(0,13,"0_to_D"))
VARIANTS=("es10","internal120","full120")
OFFICIAL=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
HANDMADE=ROOT/"02_data"/"Hand_made"

def save_csv(path, rows):
    rows=list(rows)
    assert rows, path
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader();w.writerows(rows)

class TracedMLP(MLPClassifier):
    """Hook after each original sklearn 1.8 epoch; never alters its Adam updates."""
    def set_probe(self, X, y):
        self._curve_probe_X=PixelScaler255().transform(X)
        self._curve_probe_y=np.asarray(y,dtype=int)
        self._curve_records=[]
        self._curve_start=time.perf_counter()

    def _update_no_improvement_count(self, early_stopping, X, y, sample_weight):
        # sklearn's original validation score, best coefficients and patience logic.
        super()._update_no_improvement_count(early_stopping, X, y, sample_weight)
        epoch=len(self.loss_curve_)
        pred=np.asarray(self.predict(self._curve_probe_X),dtype=int)
        truth=self._curve_probe_y
        assert len(pred)==len(truth)==640
        row={"epoch":epoch,"train_loss":float(self.loss_curve_[-1]),
             "internal_val_accuracy":float(self.validation_scores_[-1]) if early_stopping else "",
             "internal_val_loss":"",
             "heldout_oof_accuracy":float(np.mean(pred==truth)),
             "n_no_improvement":int(self._no_improvement_count)}
        for a,b,name in PAIRS:
            row[name]=int(np.count_nonzero((truth==a)&(pred==b)))
        if early_stopping and (epoch==1 or epoch%5==0):
            # Internal Validation loss is a diagnostic only, not used in stopping.
            probs=np.clip(self.predict_proba(X),1e-15,1.)
            yy=np.asarray(y)
            actual=np.argmax(yy,axis=1) if yy.ndim==2 else yy.astype(int)
            row["internal_val_loss"]=float(np.mean(-np.log(probs[np.arange(len(actual)),actual])))
        self._curve_records.append(row)

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--seed",type=int,choices=(42,43,44),required=True)
    ap.add_argument("--heldout",type=int,choices=WRITERS,required=True)
    ap.add_argument("--variant",choices=VARIANTS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    a=ap.parse_args()
    if a.out.exists():raise FileExistsError(a.out)
    a.out.mkdir(parents=True)
    off=load_official(OFFICIAL)
    writers=load_writers(HANDMADE)
    assert off["sha256"]==V3_SHA
    assert len(off["X_train"])==76800 and len(off["X_val"])==19200
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    xofficial,xwriters,augmentation,_=prepareraw(off,writers,"scale30_stroke28")
    others=[w for w in WRITERS if w!=a.heldout]
    X=np.concatenate([xofficial]+[xwriters[w] for w in others])
    y=np.concatenate([off["y_train"]]+[writers[w]["y"] for w in others])
    assert X.shape==(78720,784) and y.shape==(78720,)
    assert 0.298<=augmentation["actual_ratio"]<=0.302
    yheld=np.asarray(writers[a.heldout]["y"],dtype=int)
    es=a.variant!="full120"
    patience=10 if a.variant=="es10" else 200
    epochs=200 if a.variant=="es10" else 120
    net=TracedMLP(
        random_state=a.seed,hidden_layer_sizes=(256,),alpha=.0001,
        learning_rate_init=.001,solver="adam",activation="relu",max_iter=epochs,
        early_stopping=es,validation_fraction=.1,n_iter_no_change=patience,tol=.0001
    )
    net.set_probe(xwriters[a.heldout],yheld)
    pipe=Pipeline([("scale_0_1",PixelScaler255()),("model",net)])
    with warnings.catch_warnings(record=True) as got:
        warnings.simplefilter("always")
        started=time.perf_counter()
        pipe.fit(X,y)
        train_sec=time.perf_counter()-started
    save_csv(a.out/"epoch_curve.csv",net._curve_records)
    restored=pipe.predict(xwriters[a.heldout]).astype(int)
    save_csv(a.out/"final_heldout_predictions.csv",[
      {"seed":a.seed,"heldout":a.heldout,"index":i,"actual":LABELS[int(yheld[i])],
       "pred":LABELS[int(restored[i])]} for i in range(len(yheld))
    ])
    score=np.asarray(getattr(net,"validation_scores_",[]),dtype=float)
    best_epoch=int(np.argmax(score)+1) if es else None
    # sklearn restores weights from highest validation-score epoch, not the last epoch.
    best_curve=net._curve_records[best_epoch-1] if best_epoch else None
    matches=True
    if best_curve:
        matches=(
          abs(float(np.mean(restored==yheld))-best_curve["heldout_oof_accuracy"])<1e-12
          and all(int(np.sum((yheld==aa)&(restored==bb)))==best_curve[nn]
                  for aa,bb,nn in PAIRS)
        )
        assert matches,"Restored weights differ in OOF accuracy or key error counts from best validation epoch"
    meta={"experiment":"EXP-CMB-008","extension":"S5_Learning_Curve_Diagnostic",
       "variant":a.variant,"seed":a.seed,"heldout":a.heldout,
       "fit_epochs":int(net.n_iter_),"best_internal_val_epoch":best_epoch,
       "best_internal_val_accuracy":float(np.max(score)) if es else None,
       "final_restored_heldout_accuracy":float(np.mean(restored==yheld)),
       "last_epoch_heldout_accuracy":net._curve_records[-1]["heldout_oof_accuracy"],
       "restored_matches_best_epoch_metrics":matches if es else None,
       "start_training_samples":78720,"training_fraction_if_internal_val":.9 if es else 1.,
       "continued_patience":patience,"max_iter":epochs,"tol":.0001,
       "early_stopping":es,"validation_fraction":.1 if es else None,
       "fixed_hidden_layers":[256],"fixed_alpha":.0001,"fixed_lr":.001,
       "random_state":a.seed,"preprocess":"Official Train size30 stroke28, CoM",
       "augmentation_seed":20261116,"augmentation":augmentation,
       "original_official_sha":off["sha256"],
       "handmade_sha":{str(k):writers[k]["sha256"] for k in WRITERS},
       "fit_seconds":train_sec,
       "warnings":[str(w.message) for w in got],
       "sklearn_version":sklearn.__version__,
       "heldout_writer_used_for_training":False,
       "heldout_writer_used_for_early_stop":False,
       "official_val_used_for_early_stop":False,
       "original_test_used":False,"professor_test_used":False,"new_handmade_480_used":False}
    (a.out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("FINISHED_TRACE",json.dumps({"variant":a.variant,"seed":a.seed,
      "writer":a.heldout,"n_iter":net.n_iter_,"best_epoch":best_epoch,
      "restored_oof":meta["final_restored_heldout_accuracy"]},ensure_ascii=False),flush=True)

if __name__=="__main__":
    main()


## exp_cmb008_s5_presentation_curves

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_presentation_curves.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_presentation_curves.py)  
Git blob: `d752b0ea3a69cbbec2555d759935a5279881f8eb`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Make presentation-ready 16:9 learning curves from verified CMB008 raw CSV.

This does NOT refit models or modify HP. The same-fit 90/10 internal-val
learning curves diagnose overfitting; the ES=False selection comes from the
separate S5 heldout-writer experiment, not these traces.

CLI 인자 (정적 추출): `--csv`, `--summary`, `--s5`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_presentation_curves.py
#!/usr/bin/env python3
"""Make presentation-ready 16:9 learning curves from verified CMB008 raw CSV.

This does NOT refit models or modify HP. The same-fit 90/10 internal-val
learning curves diagnose overfitting; the ES=False selection comes from the
separate S5 heldout-writer experiment, not these traces.
"""
from __future__ import annotations
import argparse,csv,json
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

EXPECTED_EPOCHS=[1]+list(range(5,121,5))
def main():
    p=argparse.ArgumentParser()
    p.add_argument("--csv",type=Path,required=True)
    p.add_argument("--summary",type=Path,required=True)
    p.add_argument("--s5",type=Path,required=True)
    p.add_argument("--out",type=Path,required=True)
    a=p.parse_args()
    with a.csv.open(encoding="utf-8-sig",newline="") as handle:
        rows=list(csv.DictReader(handle))
    assert [int(v["epoch"]) for v in rows]==EXPECTED_EPOCHS
    assert all(int(v["n_folds"])==12 for v in rows)
    diag=json.loads(a.summary.read_text(encoding="utf-8"))
    result=json.loads(a.s5.read_text(encoding="utf-8"))
    assert diag["folds"]==12 and diag["full_actual_training_accuracy"] is True
    assert diag["independent_test_used"] is False
    assert result["stage"]=="S5_early_stopping"
    assert result["selected_early_stopping"] is False
    assert result["selected_learning_rate"]=="0.001"
    assert abs(result["scores"]["False"]["oof"]-.9221354166666667)<1e-10
    # Authoritative measurements are the original aggregated CSV, never hardcoded.
    x=[int(v["epoch"]) for v in rows]
    acc_train=[100*float(v["training_accuracy_mean"]) for v in rows]
    acc_val=[100*float(v["internal_validation_accuracy_mean"]) for v in rows]
    loss_train=[float(v["training_objective_loss_mean"]) for v in rows]
    loss_val=[float(v["internal_validation_log_loss_mean"]) for v in rows]
    best=float(diag["mean_es10_best_val_epoch"])
    stop=float(diag["mean_es10_would_stop_epoch"])
    min_i=min(range(len(x)),key=lambda k:loss_val[k])
    assert x[min_i]==10 and round(acc_train[-1],3)==99.969
    assert round(acc_val[-1],3)==96.527
    assert 30<best<stop<50
    a.out.mkdir(parents=True,exist_ok=True)
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":16,
        "axes.titlesize":20,"axes.labelsize":17,"xtick.labelsize":14,
        "ytick.labelsize":14,"legend.fontsize":13,"figure.facecolor":"white",
        "savefig.facecolor":"white"})
    BLUE="#176b89"
    ORANGE="#d06c1e"
    SLATE="#4f5965"
    RED="#bd513e"
    def style(ax):
        ax.set_xlim(0,120)
        ax.set_xticks(range(0,121,20))
        ax.grid(alpha=.20)
        ax.set_xlabel("Epoch")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    def annotate_acc(ax):
        ax.plot(x,acc_train,lw=2.7,color=BLUE,label="Training accuracy")
        ax.plot(x,acc_val,lw=2.7,color=ORANGE,label="Internal validation accuracy")
        ax.axvline(best,lw=1.7,color=SLATE,ls=":",label=f"Mean ES best epoch {best:.1f}")
        ax.axvline(stop,lw=1.7,color=SLATE,ls="--",label=f"Mean ES would-stop {stop:.1f}")
        ax.set_ylabel("Accuracy (%)")
        ax.set_ylim(88.5,101.3)
        style(ax)
    def annotate_loss(ax):
        ax.plot(x,loss_train,lw=2.7,color=BLUE,label="Training objective loss")
        ax.plot(x,loss_val,lw=2.7,color=ORANGE,label="Internal validation log-loss")
        ax.axvline(x[min_i],lw=1.7,color=RED,ls="--",label=f"Lowest mean validation loss: epoch {x[min_i]}")
        ax.set_ylabel("Loss (lower is better)")
        ax.set_ylim(0,.42)
        style(ax)
    # Main slide: two aligned panels, directly paste into PowerPoint.
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,9),gridspec_kw={"wspace":.22})
    fig.suptitle("MLP Learning Curves | S5 Early Stopping Diagnosis",fontsize=26,weight="bold",y=.965)
    ax1.set_title("A. Accuracy vs Epoch",weight="bold",pad=20)
    annotate_acc(ax1)
    ax1.legend(loc="lower right",frameon=False,fontsize=12)
    ax2.set_title("B. Loss vs Epoch",weight="bold",pad=20)
    annotate_loss(ax2)
    ax2.legend(loc="upper right",frameon=False,fontsize=12)
    fig.text(.5,.12,
             "12 folds (3 seeds x 4 held-out writers)  |  same-fit 90% Train / 10% Internal Validation",
             ha="center",color=SLATE,fontsize=15)
    fig.text(.5,.086,
             "Diagnostic: patience=200 continued to 120 epochs; vertical ES markers are counterfactual averages.",
             ha="center",color=SLATE,fontsize=12)
    fig.text(.5,.056,
             "Decision: ES=False retained from separate Writer-OOF comparison (not selected from this curve).",
             ha="center",color=SLATE,fontsize=12,style="italic")
    fig.subplots_adjust(top=.84,bottom=.22,left=.075,right=.965)
    fig.savefig(a.out/"S5_presentation_LearningCurve_16x9.png",dpi=200)
    plt.close(fig)
    # Standalone widescreen graphs for slides with large projected figures.
    fig,ax=plt.subplots(figsize=(16,9))
    fig.suptitle("S5 | Training vs Validation Accuracy",weight="bold",fontsize=28,y=.96)
    annotate_acc(ax)
    ax.legend(loc="lower right",frameon=False,fontsize=19)
    fig.text(.5,.105,
             "Same-fit 90% Train / 10% Validation  |  3 seeds x 4 writers  |  dashed ES markers: hypothetical best/stop",
             ha="center",fontsize=15,color=SLATE)
    fig.subplots_adjust(top=.865,bottom=.19,left=.10,right=.95)
    fig.savefig(a.out/"S5_presentation_Accuracy_16x9.png",dpi=200)
    plt.close(fig)
    fig,ax=plt.subplots(figsize=(16,9))
    fig.suptitle("S5 | Training vs Validation Loss",weight="bold",fontsize=28,y=.96)
    annotate_loss(ax)
    ax.legend(loc="upper right",frameon=False,fontsize=19)
    fig.text(.5,.10,
             "Validation loss rises after epoch 10; accuracy need not fall at the same time.",
             ha="center",fontsize=16,color=SLATE)
    fig.subplots_adjust(top=.865,bottom=.19,left=.10,right=.95)
    fig.savefig(a.out/"S5_presentation_Loss_16x9.png",dpi=200)
    plt.close(fig)
    print(json.dumps({"figures":3,"best":best,"stop":stop,
        "val_loss_min_epoch":x[min_i],
        "train_accuracy_epoch120":acc_train[-1],
        "val_accuracy_epoch120":acc_val[-1],
        "selected_early_stopping":result["selected_early_stopping"],
        "selected_oof":result["scores"]["False"]["oof"]},ensure_ascii=False),flush=True)
if __name__=="__main__":
    main()


## exp_cmb008_s5_train_val_curve

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_curve.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_curve.py)  
Git blob: `eb8995ed6be49aedfcd84cee241543df630e6b40`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

S5 extension: actual train-vs-validation accuracy and loss in THE SAME fit.

The 90% training subset and 10% internal validation come from sklearn 1.8.0's
actual train_test_split call, captured without modifying its returned arrays,
random generator, optimizer updates, early-stopping score, or training order.

Run a single 120-epoch fit with early_stopping=True and patience=200 so that
the same run reaches all diagnostic checkpoints. Monitor the first epoch at
which the normal patience=10 rule w

CLI 인자 (정적 추출): `--seed`, `--heldout`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_curve.py
#!/usr/bin/env python3
"""S5 extension: actual train-vs-validation accuracy and loss in THE SAME fit.

The 90% training subset and 10% internal validation come from sklearn 1.8.0's
actual train_test_split call, captured without modifying its returned arrays,
random generator, optimizer updates, early-stopping score, or training order.

Run a single 120-epoch fit with early_stopping=True and patience=200 so that
the same run reaches all diagnostic checkpoints. Monitor the first epoch at
which the normal patience=10 rule would have stopped (counterfactual stop),
without using held-out Writer OOF to choose the checkpoint.

Actual full training accuracy, validation accuracy, internal validation
unregularized cross-entropy and sklearn training objective loss are measured
at epochs 1, 5, 10, 15, ..., 120. OOF accuracy and four key confusion
directions are diagnostic only (never drive stopping or HP selection).
"""
from __future__ import annotations
import argparse,csv,json,time,warnings
from pathlib import Path
import numpy as np
import sklearn
import sklearn.neural_network._multilayer_perceptron as sklearn_mlp
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from exp_cmb008_s5_early_stopping import (
    ROOT,WRITERS,LABELS,V3_SHA,HANDMADE_SHA,
    load_official,load_writers,PixelScaler255,prepareraw
)

OFFICIAL=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
HANDMADE=ROOT/"02_data"/"Hand_made"
CONFUSIONS=((13,0,"D_to_0"),(11,8,"B_to_8"),(8,11,"8_to_B"),(0,13,"0_to_D"))
MAX_EPOCH=120
POINTS=frozenset([1]+list(range(5,MAX_EPOCH+1,5)))

def write_csv(p,rows):
    assert rows,p
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

class TrainValTraceMLP(MLPClassifier):
    """Captures sklearn's *actual* training partition from inside same fit."""
    def set_heldout(self,X,y):
        self.heldout_X=PixelScaler255().transform(X)
        self.heldout_y=np.asarray(y,dtype=int)
        self.epoch_records=[]
        self.counterfactual_patience10_stop=None
        self._internal_split_capture_count=0

    def _fit_stochastic(self,*args,**kwargs):
        original=sklearn_mlp.train_test_split
        def capture_split(*p,**k):
            out=original(*p,**k)
            self._internal_split_capture_count+=1
            self._exact_training_X=out[0]
            self._exact_training_y=self._label_binarizer.inverse_transform(out[2])
            self._exact_validation_X=out[1]
            self._exact_validation_y=self._label_binarizer.inverse_transform(out[3])
            return out
        sklearn_mlp.train_test_split=capture_split
        try:
            return super()._fit_stochastic(*args,**kwargs)
        finally:
            sklearn_mlp.train_test_split=original

    def _update_no_improvement_count(self,early_stopping,X,y,sample_weight):
        # Run sklearn's unchanged validation update and best-weight tracking.
        super()._update_no_improvement_count(early_stopping,X,y,sample_weight)
        assert early_stopping,"This experiment intentionally uses sklearn internal 10% validation."
        epoch=len(self.loss_curve_)
        if self._no_improvement_count>10 and self.counterfactual_patience10_stop is None:
            self.counterfactual_patience10_stop=int(epoch)
        if epoch not in POINTS:
            return
        assert self._internal_split_capture_count==1
        assert len(self._exact_training_X)==70848 and len(X)==7872
        # Full, actual sklearn 90% training partition, NOT a sampled proxy.
        yp_train=self.predict(self._exact_training_X)
        train_accuracy=float(np.mean(yp_train==self._exact_training_y))
        internal_accuracy=float(self.validation_scores_[-1])
        # Validation log-loss: diagnostic, not sklearn's stop criterion.
        # sklearn's training loss includes regularization; these are labelled
        # separately, and their absolute levels should not be conflated.
        probabilities=np.clip(self.predict_proba(X),1e-15,1.)
        yc=np.asarray(y,dtype=int)
        assert probabilities.shape==(7872,16)
        val_logloss=float(np.mean(-np.log(probabilities[np.arange(len(yc)),yc])))
        yp_holdout=self.predict(self.heldout_X)
        holdout_accuracy=float(np.mean(yp_holdout==self.heldout_y))
        row={"epoch":epoch,"training_accuracy":train_accuracy,
            "internal_validation_accuracy":internal_accuracy,
            "training_objective_loss":float(self.loss_curve_[-1]),
            "internal_validation_log_loss":val_logloss,
            "heldout_writer_accuracy_diagnostic_only":holdout_accuracy,
            "no_improvement_epochs":int(self._no_improvement_count)}
        for actual,pred,key in CONFUSIONS:
            row[key]=int(np.count_nonzero((self.heldout_y==actual)&(yp_holdout==pred)))
        self.epoch_records.append(row)

def main():
    a=argparse.ArgumentParser()
    a.add_argument("--seed",type=int,choices=(42,43,44),required=True)
    a.add_argument("--heldout",type=int,choices=WRITERS,required=True)
    a.add_argument("--out",type=Path,required=True)
    args=a.parse_args()
    if args.out.exists():raise FileExistsError(args.out)
    args.out.mkdir(parents=True)
    official=load_official(OFFICIAL)
    writers=load_writers(HANDMADE)
    assert official["sha256"]==V3_SHA
    for writer in WRITERS:assert writers[writer]["sha256"]==HANDMADE_SHA[writer]
    data,wdata,aug,_=prepareraw(official,writers,"scale30_stroke28")
    rest=[w for w in WRITERS if w!=args.heldout]
    X=np.concatenate([data]+[wdata[w] for w in rest])
    y=np.concatenate([official["y_train"]]+[writers[w]["y"] for w in rest])
    assert X.shape==(78720,784) and len(y)==78720
    net=TrainValTraceMLP(hidden_layer_sizes=(256,),alpha=.0001,
        solver="adam",activation="relu",learning_rate_init=.001,
        early_stopping=True,validation_fraction=.10,n_iter_no_change=200,
        tol=1e-4,max_iter=MAX_EPOCH,random_state=args.seed)
    net.set_heldout(wdata[args.heldout],writers[args.heldout]["y"])
    model=Pipeline([("scale_0_1",PixelScaler255()),("model",net)])
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter("always")
        start=time.monotonic()
        model.fit(X,y)
        elapsed=time.monotonic()-start
    assert net.n_iter_==120
    assert len(net.epoch_records)==len(POINTS)
    assert [z["epoch"] for z in net.epoch_records]==sorted(POINTS)
    assert net._internal_split_capture_count==1
    assert net.counterfactual_patience10_stop is not None,"No hypothetical stop with patience10 by 120"
    hypothetical_stop=net.counterfactual_patience10_stop
    first_best=int(np.argmax(net.validation_scores_[:hypothetical_stop])+1)
    final_best=int(np.argmax(net.validation_scores_)+1)
    # Actual stopper would have picked the highest-validation-score weights
    # at first_best and restored them, rather than this long-fit final_best.
    print("CURVE_FINISHED",json.dumps({"seed":args.seed,"heldout":args.heldout,
        "counterfactual_es10_stop":hypothetical_stop,
        "counterfactual_es10_best":first_best,
        "best_val_epoch_over_120":final_best,
        "train_accuracy_epoch120":net.epoch_records[-1]["training_accuracy"],
        "val_accuracy_epoch120":net.epoch_records[-1]["internal_validation_accuracy"]}),flush=True)
    write_csv(args.out/"same_run_train_val_curve.csv",net.epoch_records)
    m={"experiment":"EXP-CMB-008_S5","study":"TrueTrainValSameRunLearningCurve",
       "seed":args.seed,"heldout_writer":args.heldout,"total_epochs":MAX_EPOCH,
       "full_training_partition_samples":70848,"internal_validation_samples":7872,
       "actual_training_accuracy_not_subsampled":True,
       "epoch_metrics":sorted(POINTS),
       "counterfactual_es10_stop_epoch":hypothetical_stop,
       "counterfactual_es10_best_val_epoch":first_best,
       "best_validation_epoch_over_120":final_best,
       "model_init_seed":args.seed,"model_hyperparameters":{
          "hidden_layer_sizes":[256],"alpha":.0001,"activation":"relu",
          "solver":"adam","learning_rate_init":.001,
          "early_stopping":True,"validation_fraction":.1,
          "n_iter_no_change":200,"max_iter":120,"tol":.0001},
       "source_sha":{"official":official["sha256"],"handmade":{
          str(w):writers[w]["sha256"] for w in WRITERS}},
       "augmentation":aug,"sklearn_version":sklearn.__version__,
       "warnings":[str(w.message) for w in ws],"elapsed_sec":elapsed,
       "heldout_writer_used_for_fit_or_stopping":False,
       "original_test_used":False,"professor_test_used":False}
    (args.out/"manifest.json").write_text(json.dumps(m,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")

if __name__=="__main__":main()


## exp_cmb008_s5_learning_curve_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_aggregate.py)  
Git blob: `b72e2c1ce63372fc5b21fd79b9ffe18deb279e4c`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Aggregate all 36 faithful 3-seed×4-writer×3-arm learning traces.
Produce actual (not simulated) learning curves; never select HP using held-out
writer checkpoint maxima. Verify original S5 ES=True stop and final predictions.

CLI 인자 (정적 추출): `--incoming`, `--original-s5`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_learning_curve_aggregate.py
#!/usr/bin/env python3
"""Aggregate all 36 faithful 3-seed×4-writer×3-arm learning traces.
Produce actual (not simulated) learning curves; never select HP using held-out
writer checkpoint maxima. Verify original S5 ES=True stop and final predictions.
"""
from __future__ import annotations
import argparse,csv,json,statistics,math
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
ARMS=("es10","internal120","full120")
PAIRS=("D_to_0","B_to_8","8_to_B","0_to_D")
PAIRS_TEXT=("D→0","B→8","8→B","0→D")
EVAL_EPOCHS=(40,60,80,100,120)
def readcsv(p):
    with p.open(encoding="utf-8-sig",newline="") as f:return list(csv.DictReader(f))
def writecsv(p,rows):
    assert rows
    with p.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def ratio(x):return f"{100*x:.3f}%"
def acc_at_arm(rows):
    return statistics.mean(float(x["heldout_oof_accuracy"]) for x in rows)
def build():
    ap=argparse.ArgumentParser()
    ap.add_argument("--incoming",type=Path,required=True)
    ap.add_argument("--original-s5",type=Path,required=True)
    ap.add_argument("--out",type=Path,required=True)
    a=ap.parse_args()
    a.out.mkdir(parents=True,exist_ok=True)
    (a.out/"figures").mkdir(exist_ok=True)
    datasets={};metadata={};orig_replays=[]
    for variant in ARMS:
        for seed in SEEDS:
            for w in WRITERS:
                folder=a.incoming/f"cmb008-esdiag-s{seed}-w{w}-{variant}"
                assert folder.is_dir(),folder
                trace=readcsv(folder/"epoch_curve.csv")
                meta=json.loads((folder/"manifest.json").read_text(encoding="utf-8"))
                assert meta["variant"]==variant and meta["seed"]==seed and meta["heldout"]==w
                assert meta["heldout_writer_used_for_early_stop"] is False
                assert not meta["original_test_used"] and not meta["professor_test_used"]
                assert meta["fixed_hidden_layers"]==[256] and meta["fixed_alpha"]==.0001 and meta["fixed_lr"]==.001
                assert meta["augmentation"]["requested_ratio"]==.3
                assert len(trace)==meta["fit_epochs"]
                assert [int(t["epoch"]) for t in trace]==list(range(1,len(trace)+1))
                if variant=="es10":assert len(trace)<200
                else:assert len(trace)==120
                datasets[(variant,seed,w)]=trace
                metadata[(variant,seed,w)]=meta
                if variant=="es10":
                    old=a.original_s5/f"cmb008-s5-lr0.001-s{seed}"
                    assert old.is_dir(),old
                    fits=[x for x in readcsv(old/"training_time.csv") if int(x["heldout"])==w]
                    assert len(fits)==1
                    orig_n=int(fits[0]["iterations"])
                    original=[x for x in readcsv(old/"oof_predictions.csv") if int(x["writer"])==w]
                    replay=readcsv(folder/"final_heldout_predictions.csv")
                    assert len(original)==len(replay)==640
                    mismatch=sum(oldr["pred"]!=newr["pred"] for oldr,newr in zip(original,replay))
                    exact=(orig_n==meta["fit_epochs"] and mismatch==0)
                    orig_replays.append({"seed":seed,"heldout":w,"orig_stop":orig_n,
                       "replayed_stop":meta["fit_epochs"],"mismatched_predictions":mismatch,
                       "exact_replay":exact})
    assert len(datasets)==36
    # Replay must match previously published S5 before interpreting new diagnostics.
    replay_exact=sum(x["exact_replay"] for x in orig_replays)
    print("ORIGINAL_S5_REPLAY_AUDIT",json.dumps({"exact_folds":replay_exact,"total":12,"folds":orig_replays}),flush=True)
    # Different runner or BLAS floating order can change validation-based stopping;
    # never falsely claim exact replay. Preserve all mismatches for interpretation.
    # The ES=True 120-epoch continuation must exactly match ES10 until its stop:
    # only patience changes; split, initialization, Adam momentum and data are equal.
    replay_checks=[]
    for seed in SEEDS:
        for w in WRITERS:
            early=datasets[("es10",seed,w)]
            long=datasets[("internal120",seed,w)]
            assert len(early)<=len(long)
            disagreements=sum(abs(float(x["train_loss"])-float(y["train_loss"]))>1e-9
                or abs(float(x["internal_val_accuracy"])-float(y["internal_val_accuracy"]))>1e-12
                or abs(float(x["heldout_oof_accuracy"])-float(y["heldout_oof_accuracy"]))>1e-12
                or any(int(x[p])!=int(y[p]) for p in PAIRS)
                for x,y in zip(early,long))
            replay_checks.append({"seed":seed,"writer":w,"pre_stop_disagreements":disagreements})
    aligned_count=sum(x["pre_stop_disagreements"]==0 for x in replay_checks)
    print("ES10_VS_CONTINUATION_AUDIT",json.dumps({"matched_folds":aligned_count,"total":12,"folds":replay_checks}),flush=True)
    # If trajectories diverge, these are repeated seeded stochastic trajectories,
    # not an exactly paired causal stop-only comparison.
    base_sha=metadata[("es10",42,41)]["original_official_sha"]
    for m in metadata.values():
        assert m["original_official_sha"]==base_sha
        assert m["handmade_sha"]==metadata[("es10",42,41)]["handmade_sha"]
    # Per-epoch pooled 3seed×4writer (7680 held-out predictions). No peek at Test.
    aggregate=[]
    for variant in ARMS:
        upto=max(len(datasets[(variant,s,w)]) for s in SEEDS for w in WRITERS)
        for epoch in range(1,upto+1):
            avail=[datasets[(variant,s,w)][epoch-1] for s in SEEDS for w in WRITERS
                    if epoch<=len(datasets[(variant,s,w)])]
            row={"variant":variant,"epoch":epoch,"n_folds":len(avail),
              "mean_train_loss":statistics.mean(float(z["train_loss"]) for z in avail),
              "mean_oof_accuracy":acc_at_arm(avail),
              "mean_internal_validation_accuracy":(
                statistics.mean(float(z["internal_val_accuracy"]) for z in avail)
                if variant!="full120" else ""),
              "mean_internal_validation_loss":(
                statistics.mean(float(z["internal_val_loss"]) for z in avail if z["internal_val_loss"]!="")
                if variant!="full120" and any(z["internal_val_loss"]!="" for z in avail) else "")}
            for p in PAIRS:
                row[p+"_count"]=sum(int(z[p]) for z in avail) if len(avail)==12 else ""
                row[p+"_rate"]=sum(int(z[p]) for z in avail)/480 if len(avail)==12 else ""
            aggregate.append(row)
    writecsv(a.out/"learning_curve_epoch_aggregate.csv",aggregate)
    fold=[]
    for (variant,seed,w),meta in metadata.items():
        fold.append({"variant":variant,"seed":seed,"heldout":w,
          "stop_epoch":meta["fit_epochs"],"best_internal_validation_epoch":meta["best_internal_val_epoch"] or "",
          "best_internal_validation_accuracy":meta["best_internal_val_accuracy"] if meta["best_internal_val_accuracy"] is not None else "",
          "final_restored_heldout_accuracy":meta["final_restored_heldout_accuracy"],
          "last_epoch_heldout_accuracy":meta["last_epoch_heldout_accuracy"],
          "training_fraction":meta["training_fraction_if_internal_val"],
          "fit_seconds":meta["fit_seconds"]})
    writecsv(a.out/"fold_checkpoint_summary.csv",fold)
    def pooled(variant,epoch):
        ss=[datasets[(variant,s,w)][epoch-1] for s in SEEDS for w in WRITERS]
        return {"epoch":epoch,"accuracy":acc_at_arm(ss),
                "pairs":{p:sum(int(x[p]) for x in ss) for p in PAIRS}}
    checkpoints={}
    for arm in ("internal120","full120"):
        checkpoints[arm]={str(k):pooled(arm,k) for k in EVAL_EPOCHS}
    # es10 values use the already restored checkpoints after a normal sklearn fit.
    restored=[]
    for s in SEEDS:
        for w in WRITERS:
            f=a.incoming/f"cmb008-esdiag-s{s}-w{w}-es10"
            restored.extend(readcsv(f/"final_heldout_predictions.csv"))
    assert len(restored)==7680
    y=np.asarray([r["actual"] for r in restored])
    p=np.asarray([r["pred"] for r in restored])
    old_final={"accuracy":float(np.mean(y==p)),
      "pairs":{key:int(np.sum((y==aa)&(p==bb))) for aa,bb,key in (
         ("D","0","D_to_0"),("B","8","B_to_8"),("8","B","8_to_B"),("0","D","0_to_D"))}}
    original_s5_published={"accuracy":.9177083333333333,
      "pairs":{"D_to_0":77,"B_to_8":49,"8_to_B":49,"0_to_D":14}}
    print("REMEASURED_ES10_VS_ORIGINAL",json.dumps({"new":old_final,"original":original_s5_published}),flush=True)
    stop=[metadata[("es10",s,w)]["fit_epochs"] for s in SEEDS for w in WRITERS]
    best=[metadata[("es10",s,w)]["best_internal_val_epoch"] for s in SEEDS for w in WRITERS]
    summary={"experiment":"EXP-CMB-008 S5 learning curve","report_status":"evaluated",
      "original_s5_exact_replay":replay_exact==12,"original_exact_replay_folds":replay_exact,
      "pre_stop_long_continuation_exact":aligned_count==12,
      "pre_stop_aligned_folds":aligned_count,
      "counts":{"fold_variants":36,"seeds":3,"writers":4,
                 "internal_validation_samples_per_fold":7872},
      "stop_epochs":stop,"best_epochs":best,"mean_stop_epoch":statistics.mean(stop),
      "mean_best_epoch":statistics.mean(best),"restored_es10":old_final,
      "original_published_s5":original_s5_published,
      "pre_registered_diagnostic_checkpoints":checkpoints,
      "unchanged_hyperparameters":{"hidden_layer_sizes":[256],"alpha":.0001,
                                  "learning_rate_init":.001,"size_ratio":.3,"stroke_strength":.28},
      "data_source":"training and writer-held-out only","independent_test_used":False,
      "replay_s5_audit":orig_replays,"continuation_audit":replay_checks}
    (a.out/"diagnostic_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    # Charts: actual raw measurements, not smoothed or fabricated.
    def series(arm,field,complete=False):
        rows=[x for x in aggregate if x["variant"]==arm and (x["n_folds"]==12 or not complete)]
        x=[z["epoch"] for z in rows if z[field]!=""]
        y=[float(z[field]) for z in rows if z[field]!=""]
        return x,y
    colors={"full120":"#156082","internal120":"#df8425","es10":"#49865b"}
    labels={"full120":"100% train (no ES, 120 epochs)",
            "internal120":"90% train + internal val (continued to 120)",
            "es10":"90% train + ES patience 10 (until stopped)"}
    def common(ax,xlabel="Epoch",ylabel=""):
        ax.set_xlabel(xlabel);ax.set_ylabel(ylabel)
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    fig,ax=plt.subplots(figsize=(10,5.4))
    for arm in ARMS:
        xs,ys=series(arm,"mean_train_loss")
        ax.plot(xs,ys,color=colors[arm],label=labels[arm],lw=1.8)
    common(ax,ylabel="Mean training log loss");ax.set_title("Learning curve — Training loss (different train fractions)")
    fig.tight_layout();fig.savefig(a.out/"figures"/"01_training_loss.png",dpi=170);plt.close(fig)
    fig,ax=plt.subplots(figsize=(10,5.4))
    for arm in ("internal120","es10"):
        xs,ys=series(arm,"mean_internal_validation_accuracy")
        ax.plot(xs,ys,color=colors[arm],label=labels[arm],lw=1.8)
    ax.axvline(statistics.mean(best),color="gray",ls="--",label=f"ES10 mean best val epoch {statistics.mean(best):.1f}")
    ax.axvline(statistics.mean(stop),color="black",ls=":",label=f"ES10 mean stop epoch {statistics.mean(stop):.1f}")
    common(ax,ylabel="Internal validation accuracy")
    ax.set_title("Validation learning curve — best checkpoint vs stop")
    fig.tight_layout();fig.savefig(a.out/"figures"/"02_internal_validation_accuracy.png",dpi=170);plt.close(fig)
    fig,ax=plt.subplots(figsize=(10,5.4))
    for arm in ARMS:
        xs,ys=series(arm,"mean_oof_accuracy")
        ax.plot(xs,ys,color=colors[arm],label=labels[arm],lw=1.8)
    ax.axhline(old_final["accuracy"],color="gray",ls="--",label=f"ES10 restored OOF {ratio(old_final['accuracy'])}")
    common(ax,ylabel="Held-out writer accuracy (mean across available folds)")
    ax.set_title("Held-out writer learning curve (diagnostic ONLY; never choose epochs using OOF)")
    fig.tight_layout();fig.savefig(a.out/"figures"/"03_heldout_writer_oof.png",dpi=170);plt.close(fig)
    fig,axs=plt.subplots(2,2,figsize=(12,8),sharex=True)
    for ax,p,desc in zip(axs.flatten(),PAIRS,PAIRS_TEXT):
        for arm in ("full120","internal120"):
            xs,ys=series(arm,p+"_rate",complete=True)
            ax.plot(xs,[100*y for y in ys],label=labels[arm],color=colors[arm])
        ax.axhline(5,color="gray",ls="--",alpha=.8,label="Goal <5%")
        ax.set_title(desc);ax.set_ylabel("Directional error (% of actual class)")
        ax.grid(alpha=.2)
    for ax in axs[1]:ax.set_xlabel("Epoch")
    handles,lab=axs[0,0].get_legend_handles_labels()
    fig.legend(handles,lab,loc="lower center",ncol=3,fontsize=8)
    fig.suptitle("Held-out writer critical directional errors (12 folds; diagnostic only)")
    fig.tight_layout(rect=(0,.06,1,.95))
    fig.savefig(a.out/"figures"/"04_directional_errors.png",dpi=170);plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,4.7))
    n=np.arange(12);ax.scatter(n,best,label="best internal validation",s=70,color=colors["internal120"])
    ax.scatter(n,stop,label="actual ES stop",marker="x",s=85,color="black")
    ax.set_xticks(n,[f"{s}-{w}" for s in SEEDS for w in WRITERS],rotation=45)
    ax.set_xlabel("Seed–heldout writer");ax.set_ylabel("Epoch");ax.grid(alpha=.2)
    ax.legend();ax.set_title("Stop epoch vs restored best-validation checkpoint")
    fig.tight_layout();fig.savefig(a.out/"figures"/"05_best_vs_stop_epoch.png",dpi=170);plt.close(fig)
    # Evidence-first report with no hand-selected OOF-optimal checkpoint deployment.
    rows=["# EXP-CMB-008 · 13 Early Stopping Learning Curve 실측 보고","",
      "> **실측 완료:** 3 Seed × 4 Writer × 3 학습 조건 = 36개 실제 모델. 원본 S5/조건 간 재현성 차이를 모두 감사했으며 일치 여부를 반드시 아래에서 확인한다.",
      "> **제한:** 120 Epoch 연장은 진단용이다. held-out OOF 최적 Epoch를 모델 선택에 사용하지 않는다. Original Test·교수님 Test·신규 손글씨 480장 미사용.","",
      "## 비교 설계와 실험 이유","",
      "| 조건 | Training / Validation | 종료 규칙 | 무엇을 구분하는가? |",
      "|---|---|---|---|",
      "| es10 | Training Fold 90% / 내부 Val 10% | 기존 S5 ES=True, patience 10, max_iter 200 | 실제 종료 위치와 검증 최고점 |",
      "| internal120 | es10과 **동일한** 90% / 내부 Val 10% | 동일 초기값/Adam, patience 200, 120 Epoch 추적 | 종료 후 학습이 지속될 때의 변화 (OOF는 진단만) |",
      "| full120 | Training Fold 100% | ES=False, patience 200, 120 Epoch 추적 | 10% 학습데이터 제외가 결과에 미치는 영향 (검증 분할도 변함) |","",
      "## 재현성 감사 (이 결과를 해석할 때 필수)","",
      f"- **기존 S5 원본과 Stop Epoch 및 최종 OOF 예측까지 정확히 재현한 Fold:** {replay_exact}/12.",
      f"- **계측 es10과 internal120가 Stop 이전 모든 Epoch Loss·Validation·OOF가 일치한 Fold:** {aligned_count}/12.",
      "- 이 차이는 같은 입력과 Seed를 지정해도 실행 환경/부동소수점 연산 순서 등의 요소가 달라질 수 있음을 보여준다. 원인을 확정하지 않은 상태에서 **원본 S5를 정확히 재현했다거나 동일한 optimizer 경로라고 주장하지 않는다**.",
      "- 만약 일치 Fold가 12 미만이라면, 이어서 학습한 곡선과 기존 ES10은 엄격한 단일 변수 통제 비교가 아니라 독립 반복 궤적이다. OOF/Best Epoch 차이는 진단 참고 결과이며 인과적 결론·최종 HP 선택에 사용하지 않는다.",
      "- Fold별 불일치 원본은 [진단 요약 JSON](diagnostic_summary.json)의 `replay_s5_audit`와 `continuation_audit` 참조.",
      "",
      "## 실제 종료 및 최고 검증 체크포인트","",
      f"- 이번에 재측정한 ES10의 평균 **Stop Epoch {statistics.mean(stop):.2f}**, 평균 **최고 Validation Epoch {statistics.mean(best):.2f}**. 기존 S5에 발표된 평균 종료 횟수는 47.17 Epoch이다.",
      f"- **새로 재측정한 ES10**의 restored OOF = **{ratio(old_final['accuracy'])}**. 복원 모델 D→0={old_final['pairs']['D_to_0']}, B→8={old_final['pairs']['B_to_8']}, 8→B={old_final['pairs']['8_to_B']}, 0→D={old_final['pairs']['0_to_D']}건 (3 Seed×4 Writer, 실제 클래스당 480회). 기존 S5 원본은 OOF 91.771%, D→0 77건, B→8 49건, 8→B 49건, 0→D 14건이다.",
      "- 모든 Fold의 실제 Stop/Best Epoch와 OOF 복원 성능은 [fold_checkpoint_summary.csv](fold_checkpoint_summary.csv) 보존.",
      "",
      "## 실제 Learning Curve — 원본 데이터에서 생성한 PNG","",
      "![Training loss](figures/01_training_loss.png)","",
      "![Internal validation accuracy](figures/02_internal_validation_accuracy.png)","",
      "![Held-out writer OOF accuracy](figures/03_heldout_writer_oof.png)","",
      "![Directional errors](figures/04_directional_errors.png)","",
      "![Best and stop epochs](figures/05_best_vs_stop_epoch.png)","",
      "## 사전 고정한 Epoch별 진단 체크포인트","",
      "| 조건 | Epoch | OOF 정확도 | D→0 | B→8 | 8→B | 0→D |",
      "|---|---:|---:|---:|---:|---:|---:|"]
    for arm in ("internal120","full120"):
        for epoch in EVAL_EPOCHS:
            rec=checkpoints[arm][str(epoch)]
            rows.append(f"| {arm} | {epoch} | {ratio(rec['accuracy'])} | "+
              " | ".join(str(rec["pairs"][p]) for p in PAIRS)+" |")
    rows+=["","## 어떤 가설을 검증했는가?","",
       "- **H1 조기 종료:** 위 사전 고정 Epoch 60/80/100/120의 internal120 OOF와 각 오류 방향을 원본 ES10 복원 모델과 비교해, 학습 지속이 실질적인 개선을 보이는지 판단한다. **가장 잘 나온 OOF Epoch를 사후 선택하지 않는다.**",
       "- **H2 내부 검증/Writer 일반화 불일치:** Figure 2의 내부 Validation Accuracy 최고 구간과 Figure 3의 보류 Writer OOF 변화, Figure 4의 D→0을 나란히 비교한다.",
       "- **H3 10% 학습데이터 분리:** 동일 Epoch에서 full120(100% 학습)과 internal120(90% 학습)을 비교한다. 내부 검증 사용/랜덤 미니배치의 차이까지 완벽히 분리한 인과 실험은 아니므로 '데이터 감소만의 효과'로 단정하지 않는다.",
       "- **후속 의사결정:** Early Stopping 설정의 최종 채택/기각은 새 모델을 OOF 최고 Epoch에 맞춰 고르는 대신, Training 내부 validation 기반으로 미리 정한 변경안에서 다시 검증해야 한다.",
       "- **과적합 주의:** 개발용 4 Writer를 S1~S5에서 반복 사용했으므로 추가 OOF도 독립 일반화 측정이 아니며, Original Test 및 교수님 Test는 그대로 봉인했다.",
       "",
       "## 데이터와 재현","",
       "- [원본 Epoch별 관측 CSV](learning_curve_epoch_aggregate.csv) · [진단 요약 JSON](diagnostic_summary.json) · [Fold별 Stop/Best](fold_checkpoint_summary.csv)",
       "- 기존 S5와 개별 Fold의 정확한 재현 여부를 `diagnostic_summary.json`에서 공개했다. 불일치한 경우 이전 S5 선택 성능을 새로운 곡선의 정확한 마지막 점과 혼동하지 않는다.",
       ""]
    (a.out/"report.md").write_text("\n".join(rows),encoding="utf-8")
    print("CURVES_VERIFIED",json.dumps({"mean_stop":statistics.mean(stop),"mean_best":statistics.mean(best),
      "original_oof":old_final["accuracy"],"checkpoints":checkpoints},ensure_ascii=False),flush=True)
if __name__=="__main__":
    build()


## exp_cmb008_s5_train_val_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_aggregate.py)  
Git blob: `491294a250899a59a162f09883ea32d81faf89cf`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Generate actual same-fit train-vs-validation Accuracy and Loss learning curves.

CLI 인자 (정적 추출): `--incoming`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_train_val_aggregate.py
#!/usr/bin/env python3
"""Generate actual same-fit train-vs-validation Accuracy and Loss learning curves."""
from __future__ import annotations
import argparse,csv,json,statistics
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
POINTS=[1]+list(range(5,121,5))
FIELDS=("training_accuracy","internal_validation_accuracy",
        "training_objective_loss","internal_validation_log_loss",
        "heldout_writer_accuracy_diagnostic_only")
PAIRS=("D_to_0","B_to_8","8_to_B","0_to_D")
def readcsv(path):
    with path.open(encoding="utf-8-sig",newline="") as f:return list(csv.DictReader(f))
def writecsv(path,rows):
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def main():
    p=argparse.ArgumentParser()
    p.add_argument("--incoming",type=Path,required=True)
    p.add_argument("--out",type=Path,required=True)
    a=p.parse_args()
    a.out.mkdir(parents=True,exist_ok=True)
    figdir=a.out/"figures";figdir.mkdir(exist_ok=True)
    curves={};manifests={}
    for seed in SEEDS:
        for writer in WRITERS:
            folder=a.incoming/f"cmb008-trainval-s{seed}-w{writer}"
            assert folder.is_dir(),folder
            m=json.loads((folder/"manifest.json").read_text(encoding="utf-8"))
            vals=readcsv(folder/"same_run_train_val_curve.csv")
            assert m["seed"]==seed and m["heldout_writer"]==writer
            assert m["total_epochs"]==120 and m["full_training_partition_samples"]==70848
            assert m["internal_validation_samples"]==7872
            assert m["actual_training_accuracy_not_subsampled"]
            assert not m["original_test_used"] and not m["professor_test_used"]
            assert [int(row["epoch"]) for row in vals]==POINTS
            for row in vals:
                assert 0<=float(row["training_accuracy"])<=1
                assert 0<=float(row["internal_validation_accuracy"])<=1
                assert float(row["training_objective_loss"])>0
                assert float(row["internal_validation_log_loss"])>0
            curves[(seed,writer)]=vals
            manifests[(seed,writer)]=m
    assert len(curves)==12
    source=manifests[(42,41)]["source_sha"]
    assert all(m["source_sha"]==source for m in manifests.values())
    rows=[]
    for i,epoch in enumerate(POINTS):
        observed=[curves[k][i] for k in sorted(curves)]
        z={"epoch":epoch,"n_folds":12}
        for key in FIELDS+PAIRS:
            values=[float(x[key]) for x in observed]
            z[key+"_mean"]=statistics.mean(values)
            z[key+"_std"]=statistics.stdev(values)
            if key in PAIRS:
                z[key+"_total"]=int(sum(values))
                z[key+"_rate"]=sum(values)/480
        z["train_minus_val_accuracy_pp"]=100*(z["training_accuracy_mean"]-z["internal_validation_accuracy_mean"])
        rows.append(z)
    writecsv(a.out/"same_run_train_val_curve_aggregate.csv",rows)
    stops=[x["counterfactual_es10_stop_epoch"] for x in manifests.values()]
    best=[x["counterfactual_es10_best_val_epoch"] for x in manifests.values()]
    stop_mean=statistics.mean(stops);best_mean=statistics.mean(best)
    fold_summary=[{"seed":s,"heldout_writer":w,
      "es10_stop_epoch_same_fit":manifests[(s,w)]["counterfactual_es10_stop_epoch"],
      "es10_best_val_epoch_same_fit":manifests[(s,w)]["counterfactual_es10_best_val_epoch"],
      "best_val_epoch_over_120":manifests[(s,w)]["best_validation_epoch_over_120"]}
      for s in SEEDS for w in WRITERS]
    writecsv(a.out/"per_fold_best_stop.csv",fold_summary)
    val_loss_min=min(rows,key=lambda z:z["internal_validation_log_loss_mean"])
    val_acc_max=max(rows,key=lambda z:z["internal_validation_accuracy_mean"])
    x=[z["epoch"] for z in rows]
    train=[100*z["training_accuracy_mean"] for z in rows]
    val=[100*z["internal_validation_accuracy_mean"] for z in rows]
    tloss=[z["training_objective_loss_mean"] for z in rows]
    vloss=[z["internal_validation_log_loss_mean"] for z in rows]
    oof=[100*z["heldout_writer_accuracy_diagnostic_only_mean"] for z in rows]
    def guide(ax):
        ax.axvline(best_mean,color="#8b5e34",ls=":",lw=1.4,label=f"Mean ES10 best-val epoch {best_mean:.1f}")
        ax.axvline(stop_mean,color="#555555",ls="--",lw=1.4,label=f"Mean ES10 would-stop epoch {stop_mean:.1f}")
        ax.set_xlim(0,120);ax.grid(alpha=.23)
    def export(fig,name):
        fig.tight_layout();fig.savefig(figdir/name,dpi=170,bbox_inches="tight");plt.close(fig)
    fig,ax=plt.subplots(figsize=(11,5.8))
    ax.plot(x,train,"o-",ms=3,lw=2,color="#167c86",label="Full training partition Accuracy (70,848)")
    ax.plot(x,val,"o-",ms=3,lw=2,color="#d47620",label="Internal validation Accuracy (7,872)")
    guide(ax);ax.set(xlabel="Epoch",ylabel="Accuracy (%)",title="S5 same-fit Learning Curve — Training vs Validation Accuracy")
    ax.legend(fontsize=8);export(fig,"01_training_vs_validation_accuracy.png")
    fig,ax=plt.subplots(figsize=(11,5.8))
    ax.plot(x,tloss,"o-",ms=3,lw=2,color="#167c86",label="Training Objective Loss")
    ax.plot(x,vloss,"o-",ms=3,lw=2,color="#d47620",label="Validation Cross-Entropy Loss")
    guide(ax);ax.set(xlabel="Epoch",ylabel="Loss (lower is better)",title="S5 same-fit Learning Curve — Training vs Validation Loss")
    ax.legend(fontsize=8);export(fig,"02_training_vs_validation_loss.png")
    fig,ax=plt.subplots(figsize=(11,5.8))
    ax.plot(x,tloss,"o-",ms=3,lw=2,color="#167c86",label="Training objective loss")
    ax.plot(x,vloss,"o-",ms=3,lw=2,color="#d47620",label="Validation cross-entropy")
    ax.axvline(val_loss_min["epoch"],color="#a12f34",ls="-.",lw=1.4,label=f"Lowest mean validation loss: epoch {val_loss_min['epoch']}")
    guide(ax);ax.set(xlim=(5,120),ylim=(0,max(vloss[1:])*1.13),xlabel="Epoch",ylabel="Loss",
             title="Overfitting diagnostic — Validation loss rises")
    ax.legend(fontsize=8);export(fig,"03_overfit_loss_zoom.png")
    fig,ax=plt.subplots(figsize=(10,5.4))
    ax.plot(x,[z["train_minus_val_accuracy_pp"] for z in rows],"o-",ms=3,lw=2,color="#4b6c91",label="Training minus validation accuracy")
    guide(ax);ax.set(xlabel="Epoch",ylabel="Gap (percentage points)",title="Generalization gap by Epoch")
    ax.legend(fontsize=8);export(fig,"04_generalization_accuracy_gap.png")
    fig,ax=plt.subplots(figsize=(11,5.6))
    ax.plot(x,val,"o-",ms=3,lw=2,label="Internal validation Accuracy",color="#d47620")
    ax.plot(x,oof,"o-",ms=3,lw=2,label="Writer-held-out OOF (diagnostic only)",color="#7159a2")
    guide(ax);ax.set(xlabel="Epoch",ylabel="Accuracy (%)",title="Internal Validation vs Writer-held-out OOF")
    ax.legend(fontsize=8);export(fig,"05_validation_vs_oof.png")
    fig,ax=plt.subplots(figsize=(10,5.5))
    for name,label in zip(PAIRS,("D→0","B→8","8→B","0→D")):
        ax.plot(x,[100*z[name+"_rate"] for z in rows],"o-",ms=2,lw=1.6,label=label)
    ax.axhline(5,color="#555555",ls=":",label="Goal: <5%")
    ax.set(xlabel="Epoch",ylabel="Directional error rate (%)",title="Writer-held-out errors: four sensitive directions")
    ax.grid(alpha=.23);ax.legend(fontsize=8);export(fig,"06_writer_directional_errors.png")
    meta={"experiment":"EXP-CMB-008 Same-fit S5 Learning Curve",
      "folds":12,"epochs_measured":POINTS,"training_samples_per_fold":70848,
      "internal_val_samples_per_fold":7872,"full_actual_training_accuracy":True,
      "mean_es10_would_stop_epoch":stop_mean,"mean_es10_best_val_epoch":best_mean,
      "best_aggregate_validation_loss_epoch":val_loss_min["epoch"],
      "best_aggregate_validation_accuracy_epoch":val_acc_max["epoch"],
      "study_diagnostic_only":True,"independent_test_used":False}
    (a.out/"diagnostic_summary.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    def percent(z):return f"{100*z:.3f}%"
    report=["# EXP-CMB-008 · 16 Same-fit Training Accuracy / Validation Accuracy","",
      "이번 그래프의 Training Accuracy와 Validation Accuracy는 동일한 모델·Epoch·실행에서 실제 70,848개 Train과 7,872개 Internal Validation으로 계산했다. 각 Epoch별 12 Fold 평균이며, Training Accuracy는 부분 샘플이 아니라 전체 학습 집합이다.","",
      "## 1. Training vs Validation Accuracy","",
      "![Training and Validation Accuracy](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/figures/01_training_vs_validation_accuracy.png)","",
      "## 2. Training vs Validation Loss","",
      "![Training and Validation Loss](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/figures/02_training_vs_validation_loss.png)","",
      "![Loss overfit zoom](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/figures/03_overfit_loss_zoom.png)","",
      "## 3. Generalization gap","",
      "![Accuracy gap](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/figures/04_generalization_accuracy_gap.png)","",
      "## 4. Internal Validation vs Writer-held-out OOF","",
      "![Validation and OOF](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/figures/05_validation_vs_oof.png)","",
      "![Four confusion directions](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/figures/06_writer_directional_errors.png)","",
      "## 실제 측정값","",
      "| Epoch | Training Accuracy | Internal Validation Accuracy | Training Objective Loss | Validation Cross-Entropy Loss | Train-Val gap |",
      "|---:|---:|---:|---:|---:|---:|"]
    for z in rows:
        report.append(f"| {z['epoch']} | {percent(z['training_accuracy_mean'])} | {percent(z['internal_validation_accuracy_mean'])} | {z['training_objective_loss_mean']:.6f} | {z['internal_validation_log_loss_mean']:.6f} | {z['train_minus_val_accuracy_pp']:+.3f} pp |")
    report+=["","## 동일 실행에서 재구성한 Early Stopping 판단",
      f"- patience=10일 때의 동일 실행 경로에서 계산한 평균 중지 Epoch: **{stop_mean:.2f}**, 평균 복원 Best Validation Epoch: **{best_mean:.2f}**.",
      f"- 관측된 5-Epoch 간격에서 평균 Validation Loss 최저점: **{val_loss_min['epoch']} Epoch**.",
      f"- 평균 Validation Accuracy 최고 관측점: **{val_acc_max['epoch']} Epoch**.",
      "- Early Stopping은 Accuracy 기준으로 체크포인트를 선택하므로, Loss 기준 과적합 시작점과 같은 Epoch일 필요가 없다.",
      "- Training Objective Loss에는 L2 규제가 포함되며 Validation Log-loss는 예측 확률의 Cross-Entropy이므로 절대 수준을 동일 지표처럼 단순 비교하지 않는다.",
      "- 같은 Fold의 같은 학습 경로에서 Train/Val를 추적한 자료여서 서로 다른 실행 결과를 정렬해 발생하는 착시를 줄였다.",
      "- OOF는 오직 진단용이다. OOF의 최고 Epoch를 선택하면 새로운 Writer 데이터에 대한 선택 과적합을 유발할 수 있다.",
      "- 독립 Test, 교수님 Test, 신규 480장 미사용. ES=False는 아직 최종 재검증 전 잠정 선택값이다.",
      "",
      "## 원본", "- [집계 CSV](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/same_run_train_val_curve_aggregate.csv)",
      "- [Fold별 Best/Stop Epoch](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/per_fold_best_stop.csv)",
      "- [요약 JSON](../../results/Hyperparameter_Optimization/S5_SameFit_TrainVal/diagnostic_summary.json)",""]
    (a.out/"report.md").write_text("\n".join(report),encoding="utf-8")
    print("SAME_FIT_CURVES_DONE",json.dumps({"mean_es10_best":best_mean,
      "mean_es10_stop":stop_mean,"loss_min_epoch":val_loss_min["epoch"],
      "end_training_accuracy":rows[-1]["training_accuracy_mean"],
      "end_internal_val_accuracy":rows[-1]["internal_validation_accuracy_mean"]}),flush=True)

if __name__=="__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")